# Cartes d'identité des fichiers CSV

Pour chaque CSV du dossier `datas`, génère une carte d'identité Markdown dans
`artifacts/ingestion/identities/<AAAAMMJJ>/<fichier>_<HHMMSS>.md` :
source, taille, horodatage du traitement, encodage, dimensions, types inférés des colonnes
et statistiques des colonnes numériques et dates.

In [26]:
import csv
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from charset_normalizer import from_path

DATA_DIR = Path("datas")
OUTPUT_DIR = Path("artifacts/ingestion/identities")

## Fonctions de détection

In [27]:
def detect_charset(path: Path) -> str:
    match = from_path(path).best()
    return match.encoding.upper().replace("_", "-") if match else "UNKNOWN"


def detect_separator(path: Path, encoding: str) -> str:
    with path.open(encoding=encoding) as f:
        sample = f.read(64 * 1024)
    return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter


def human_size(size: int) -> str:
    for unit in ["octets", "Ko", "Mo", "Go"]:
        if size < 1024 or unit == "Go":
            return f"{size:.2f} {unit}"
        size /= 1024

## Inférence des types

| Type | Règle |
|---|---|
| IDENTITY | texte au format code : lettres + chiffres (`MACH-01`, `INC-000001`, `OP1002`) |
| DATETIME / DATE / TIME | texte entièrement lisible avec un format de date connu |
| BOOLEAN | entier ne contenant que 0 et 1 |
| INTEGER / FLOAT | type numérique détecté par pandas |
| CATEGORY | texte avec 20 valeurs distinctes au plus |
| TEXT | autre texte |

In [28]:
IDENTITY_PATTERN = re.compile(r"^[A-Z]+-?\d+$")
DATE_FORMATS = {
    "DATETIME": ["%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M"],
    "DATE": ["%Y-%m-%d", "%d/%m/%Y"],
    "TIME": ["%H:%M:%S", "%H:%M"],
}
NUMERIC_TYPES = {"INTEGER", "FLOAT"}
TEMPORAL_TYPES = {"DATETIME", "DATE", "TIME"}


def parse_dates(values: pd.Series):
    for kind, formats in DATE_FORMATS.items():
        for fmt in formats:
            parsed = pd.to_datetime(values, format=fmt, errors="coerce")
            if parsed.notna().all():
                return kind, parsed
    return None, None


def infer_type(col: pd.Series) -> str:
    values = col.dropna()
    if pd.api.types.is_bool_dtype(col):
        return "BOOLEAN"
    if pd.api.types.is_integer_dtype(col):
        return "BOOLEAN" if set(values.unique()) <= {0, 1} else "INTEGER"
    if pd.api.types.is_float_dtype(col):
        return "FLOAT"
    values = values.astype(str)
    if values.str.match(IDENTITY_PATTERN).all():
        return "IDENTITY"
    kind, _ = parse_dates(values)
    if kind:
        return kind
    return "CATEGORY" if values.nunique() <= 20 else "TEXT"

## Statistiques par colonne

In [29]:
def column_stats(col: pd.Series, col_type: str) -> dict:
    values = col.dropna()
    if col_type in NUMERIC_TYPES:
        return {
            "min": values.min(),
            "max": values.max(),
            "mean": round(values.mean(), 3),
            "median": values.median(),
        }
    _, parsed = parse_dates(values.astype(str))
    fmt = {"DATETIME": "%Y-%m-%d %H:%M:%S", "DATE": "%Y-%m-%d", "TIME": "%H:%M:%S"}[col_type]
    return {"min": parsed.min().strftime(fmt), "max": parsed.max().strftime(fmt)}

## Construction de la carte d'identité

In [30]:
def build_identity(path: Path) -> tuple[str, datetime]:
    begin = datetime.now()
    charset = detect_charset(path)
    sep = detect_separator(path, charset)
    df = pd.read_csv(path, sep=sep, encoding=charset)
    types = {name: infer_type(df[name]) for name in df.columns}
    stats = {name: column_stats(df[name], t) for name, t in types.items() if t in NUMERIC_TYPES | TEMPORAL_TYPES}
    end = datetime.now()

    lines = [
        f"# Carte d'identité : {path.name}",
        "",
        f"- Source: {path.resolve()}",
        f"- Filesize: {human_size(path.stat().st_size)}",
        f"- ProcessBeginAt: {begin:%H:%M:%S}",
        f"- ProcessEndAt: {end:%H:%M:%S}",
        f"- Charset: {charset}",
        f"- Separator: {sep!r}",
        f"- Rows Number: {len(df)}",
        f"- Cols Number: {len(df.columns)}",
        "",
        "## Colonnes",
        "",
        "| Colname | Infered Type |",
        "| --- | --- |",
        *[f"| {name} | {t} |" for name, t in types.items()],
        "",
        "## Colonnes numériques et dates",
        "",
        "| Colname | Min | Max | Mean | Median |",
        "| --- | --- | --- | --- | --- |",
        *[
            f"| {name} | {s['min']} | {s['max']} | {s.get('mean', '')} | {s.get('median', '')} |"
            for name, s in stats.items()
        ],
    ]
    return "\n".join(lines) + "\n", begin


def write_identity(path: Path) -> Path:
    content, begin = build_identity(path)
    out_dir = OUTPUT_DIR / f"{begin:%Y%m%d}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f"{path.stem}_{begin:%H%M%S}.md"
    out_file.write_text(content, encoding="utf-8")
    return out_file

## Génération

In [32]:
for csv_file in sorted(DATA_DIR.glob("*.csv")):
    print(write_identity(csv_file))

artifacts\ingestion\identities\20261120\releves_incidents_152328.md
artifacts\ingestion\identities\20261120\telemetry_152328.md
